# 01 — Intro to API Calls

See [README.md](README.md) for the full exercise description and context. Fill in each TODO cell, run it, and check the output makes sense before moving to the next one.

In [12]:
from typing import Literal, TypeAlias

from anthropic import Anthropic
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

anthropic_client = Anthropic()  # reads ANTHROPIC_API_KEY from env
openai_client = OpenAI()  # reads OPENAI_API_KEY from env

Message: TypeAlias = dict[Literal["role", "content"], str]
Messages: TypeAlias = list[Message]


## 1. Basic call

Send a `system` + `user` message to a model, print both the raw response object and just the text content. Do it for an Anthropic model and an OpenAI model — notice Anthropic takes `system` as a separate argument, not a message in the list.

In [ ]:
messages: Messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France?"},
]

# TODO: call anthropic_client.messages.create(...) with a "claude-*" model.
# Remember: system prompt goes in the `system=` kwarg, and `messages` should
# only contain the user message (no "system" role entry for Anthropic).
anthropic_response = anthropic_client.messages.create(
    model="claude-3-5-sonnet-20240620",
    system=messages[0]["content"],
    messages=[{"role": "user", "content": messages[1]["content"]}],
)
print(anthropic_response)


In [14]:
# TODO: call openai_client.chat.completions.create(...) with a "gpt-*" model.
# Here the system message DOES go inside `messages`.

openai_response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
)
print(openai_response)


ChatCompletion(id='chatcmpl-EM9htYQuw2SxZwlpWp5gLMc2kPy71', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The capital of France is Paris.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1788949237, model='gpt-4o-mini-2024-07-18', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint='fp_d2060853a2', usage=CompletionUsage(completion_tokens=7, prompt_tokens=24, total_tokens=31, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None)))


## 2. Roles

Try a `system` prompt that changes tone/behavior, then (Anthropic only) add a trailing `assistant`-role message as a prefill. Confirm OpenAI ignores a trailing assistant message; Anthropic continues from it.

In [16]:
from logging import RootLogger


messages: Messages = [
    {"role": "system", "content": "You are a pirate. Stay in character."},
    {"role": "user", "content": "Tell me about the weather today."},
    {"role": "assistant", "content": "Arr, here is my name first "},
    {"role": "user", "content": "sure, what is it?"},
    ]

# TODO: try this against both clients and compare how the assistant prefill
# is (or isn't) respected

openai_response= openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
)
print(openai_response.choices[0].message.content)

openai_response= openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
)
print(openai_response.choices[0].message.content)



I be known as Captain Chatbeard, scourge of the seas! Now, what be ye wantin’ to know about the weather today, matey? Speak fast, for I’ve got treasures to chase and sails to fill!
Ahoy there! I be known as Captain ChatGPT, master of the seas of knowledge! Now, what be yer next question, matey? Be ye lookin’ for treasure, tales of adventure, or news of the weather? Spit it out!


## 3. `generate_response` wrapper

One function that picks the right client based on the model name and returns just the text. Every later exercise will import this.

In [18]:
def call_anthropic(model: str, messages: Messages, temperature: float, max_tokens: int) -> str:
    """Split the system message out of `messages`, call
    anthropic_client.messages.create(...), return response.content[0].text."""
    system_prompt = next((m["content"] for m in messages if m["role"] == "system"), None)
    other_messages = [m for m in messages if m["role"] != "system"]

    kwargs = {
        "model": model,
        "messages": other_messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    if system_prompt is not None:
        kwargs["system"] = system_prompt

    response = anthropic_client.messages.create(**kwargs)
    return response.content[0].text


def call_openai(model: str, messages: Messages, temperature: float, max_tokens: int) -> str:
    """TODO: call openai_client.chat.completions.create(...),
    return response.choices[0].message.content."""
    
    kwargs = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    response = openai_client.chat.completions.create(**kwargs)
    return response.choices[0].message.content



def generate_response(
    model: str,
    messages: Messages,
    temperature: float = 1.0,
    max_tokens: int = 1000,
) -> str:
    """TODO: dispatch to call_anthropic or call_openai based on `model`
    (e.g. names starting with "claude" go to Anthropic, else OpenAI)."""
    if model.startswith("claude"):
        return call_anthropic(model, messages, temperature, max_tokens)
    else:
        return call_openai(model, messages, temperature, max_tokens)


In [21]:
# Sanity check — should print a real completion from each model.
messages: Messages = [
    {"role": "system", "content": "You are a helpful assistant who answers all questions in limericks."},
    {"role": "user", "content": "Who are you, and who were you designed by?"},
]
for model in ["gpt-4o-mini", "claude-sonnet-4-5"]:
    print(f"MODEL: {model}")
    print(generate_response(model=model, messages=messages, max_tokens=50))
    print()

#we have not given an anthropic key yet, which is why the error

MODEL: gpt-4o-mini
I am an AI, quite clever and spry,  
Designed by OpenAI, oh my!  
With knowledge to share,  
In rhymes, I declare,  
To answer your queries, I fly!

MODEL: claude-sonnet-4-5


TypeError: Messages.create() got an unexpected keyword argument 'temperature'

## 4. (Optional) retry with exponential backoff

Wrap `generate_response` so a rate-limit error triggers a sleep-and-retry with increasing backoff instead of crashing. Low value in isolation, but you'll want it once you're running batches of eval questions.

In [ ]:
import time


def retry_with_exponential_backoff(
    func,
    max_retries: int = 20,
    initial_sleep_time: float = 1.0,
    backoff_factor: float = 1.5,
):
    """TODO: wrap `func` so a rate-limit error sleeps (growing by
    `backoff_factor` each retry) and retries, up to `max_retries` times.
    Any other exception should raise immediately."""

    def wrapper(*args, **kwargs):
        raise NotImplementedError

    return wrapper


generate_response_safe = retry_with_exponential_backoff(generate_response)
